# Dopamine PINN (2D, JAX) — Complete Colab Pipeline

Companion notebook for paper **B3**: *Physics-Informed Neural Networks for Modeling Dopamine Neurotransmitter Diffusion in Synaptic Clefts* (PLOS Computational Biology).

**Implementation:** pure JAX with **Flax NNX** (network), **Optax** (Adam), and **jaxopt** (L-BFGS). JIT compilation makes this ~2-5× faster than the DeepXDE/PyTorch baseline preserved in `dopamine_PINN_deepxde.ipynb`.

**Governing PDE (2D):**
$$\partial_t C = D\,\nabla^2 C - k C, \quad (x, y) \in (-L/2, L/2)^2,\ t \in (0, T]$$
with a radially symmetric Gaussian initial condition and zero-flux Neumann boundary conditions on all four edges of the square domain.

## What this notebook does

1. Forward 2D PINN: solve the planar dopamine reaction-diffusion PDE, compare against the 2D analytical solution and a 2D finite-difference reference solver
2. Inverse 2D PINN: recover $D$ and $k$ from noisy synthetic observations sampled over $(x, y, t)$, benchmarked against an FD-solver reference ('oracle') estimator
3. Auto-fill the LaTeX manuscript with the computed numbers
4. Download all artifacts

## Runtime

**Select a GPU runtime**: *Runtime → Change runtime type → GPU* (T4 is sufficient). Expect roughly 2-3 hours total at full fidelity (T = 50 ms; five-seed ensemble plus three sweeps).

## Toggle flags (in Cell 4)

| Flag | Effect |
|---|---|
| `QUICK = True` | Reduces iterations / collocation points for ~3-minute sanity check |
| `AUTO_FILL = True` | Prompts for `B3_Dopamine_PINN_Paper.tex` upload and substitutes the [TBD] markers |

## 0. Install dependencies

Installs JAX + matching CUDA plugin + Flax NNX + Optax + jaxopt all together (so the `jax` and `jaxlib`/CUDA-plugin versions stay in lockstep — installing only Flax can otherwise upgrade `jax` past what the preinstalled CUDA plugin supports and trigger `PJRT_FFI_UserData_Add_Args size: expected 48, got 40`).

**After this cell finishes, the kernel will auto-restart.** Then click **Runtime → Run all** to continue from Cell 2 onwards. Do NOT re-run this install cell after the restart — that wastes 2 minutes.

In [ ]:
import os
from pathlib import Path

# /tmp persists across kernel restarts within the same Colab VM session.
# The sentinel name is versioned: bump it whenever the pins below change,
# so a VM that already ran an older install cell re-installs.
SENTINEL = Path('/tmp/jax_pinn_deps_ready_v2')

if not SENTINEL.exists():
    # Pinned, mutually compatible versions. Do NOT use a bare --upgrade:
    # the newest flax (0.12.9) imports jax.experimental.hijax.HiPrimitive,
    # which the newest jax (0.11.2) removed, so an unpinned upgrade fails
    # at `from flax import nnx` with an AttributeError.
    # Flax NNX + Optax for PINN training, jaxopt for L-BFGS, Blackjax for HMC.
    rc = os.system('pip -q install '
                   '"jax[cuda12]==0.10.2" '
                   '"flax==0.12.5" '
                   '"optax>=0.2.3" '
                   '"jaxopt>=0.8" '
                   '"blackjax>=1.2.0" '
                   'scipy matplotlib')
    if rc != 0:
        raise RuntimeError('pip install failed - see the output above.')

    SENTINEL.touch()
    print('Install complete. Restarting kernel to load the new JAX libraries...')
    print('After the kernel restart, click Runtime -> Run all again.')
    print('This cell will be a no-op on the second run.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed in this VM session - skipping.')

In [ ]:
import os, time, json, re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc                    # for Latin Hypercube Sampling

import jax
import jax.numpy as jnp
from flax import nnx
import optax
from jaxopt import LBFGS
from scipy.interpolate import RegularGridInterpolator

# float64 is required for k-parameter recovery in the inverse problem.
jax.config.update('jax_enable_x64', True)

SEED = 1234
np.random.seed(SEED)

FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

print(f'JAX version    : {jax.__version__}')
import flax
print(f'Flax version   : {flax.__version__}')
print(f'JAX devices    : {jax.devices()}')
print(f'Default backend: {jax.default_backend()}')
print(f'Default dtype  : float64 (required for inverse-problem accuracy)')
print(f'Sampling       : Latin Hypercube (interior collocation)')

## 1. Toggles and physical parameters

In [ ]:
# Toggles
QUICK        = False    # ~2-minute sanity check; not paper-quality
AUTO_FILL    = False    # upload B3.tex, substitute metrics, download
NOISE_SWEEP  = True     # additional inverse-problem sweep at 5 noise levels
                        # (~30-40 min extra on T4 -- set False to skip)

# Physical parameters (striatal dopamine; Cragg & Rice 2004, Trends Neurosci 27:270)
D_TRUE = 0.32     # effective diffusion coefficient (mu_m^2 / ms):
                  #   D* = D / lambda^2 = 0.763 / 1.54^2 = 0.322  (Cragg & Rice 2004, Box 1;
                  #   D* = D / lambda^2 relation: Nicholson & Phillips 1981)
K_TRUE = 0.020    # linearised DAT reuptake rate (1 / ms):
                  #   k' = Vmax / Km = (4.1 uM/s) / (0.21 uM) ~ 20 1/s = 0.020 1/ms
                  #   (Cragg & Rice 2004; the value used in their simulations)
L      = 5.0      # domain side length (mu_m); domain is [-L/2, L/2]^2.
                  #   Neighbouring DA synapse at r = 5 mu_m (Cragg & Rice 2004, Fig 2);
                  #   the zero-flux walls at +-L/2 are the symmetry planes between
                  #   two release sites 5 mu_m apart.
T      = 50.0     # simulation / observation window (ms): ~ 1/k = one DAT uptake
                  #   time constant at k' = 20 1/s (Cragg & Rice 2004). A 20 ms
                  #   window contains too little decay to identify k.
SIGMA  = 0.5      # release pulse width (mu_m) -- modelling choice
C0     = 1.0      # peak concentration scale (mu_M) -- normalisation; the PDE is
                  #   linear, so recovered (D, k) are independent of C0

# Deliberately offset initial guesses for the inverse problem
# (same relative offsets as before: D -6.25%, k -20%)
D_INIT = 0.30
K_INIT = 0.016

# Network and training hyperparameters
N_DOMAIN    = 10_000
N_BOUNDARY  = 400        # 100 per edge x 4 edges
N_INITIAL   = 400        # over the 2D square
N_TEST      = 5_000
LAYERS      = [3] + [64] * 4 + [1]   # input is (x, y, t)
ADAM_ITERS  = 20_000
LBFGS_ITERS = 2_000      # bumped from 500; needed to converge k tightly
LR          = 1e-3
N_CHUNK     = 50

# Inverse-problem observation count.
N_OBS = 400

# Noise sweep levels (as percentage of peak C0).
# Skipped entirely if NOISE_SWEEP=False.
NOISE_LEVELS = [0.0, 1.0, 2.0, 5.0, 10.0]

if QUICK:
    N_DOMAIN, N_BOUNDARY, N_INITIAL, N_TEST = 1_000, 80, 80, 500
    ADAM_ITERS  = 1_000
    LBFGS_ITERS = 200
    N_CHUNK     = 25
    N_OBS = 80
    NOISE_LEVELS = [0.0, 2.0, 10.0]   # 3 levels for QUICK
    print('[QUICK MODE] Reduced hyperparameters - not paper-quality.')

assert ADAM_ITERS % N_CHUNK == 0, 'ADAM_ITERS must be a multiple of N_CHUNK'

print(f'D = {D_TRUE} mu_m^2/ms, k = {K_TRUE} 1/ms')
print(f'Domain: [-{L/2}, {L/2}]^2 mu_m, T = {T} ms')
print(f'Architecture: {LAYERS}, Adam iters: {ADAM_ITERS} ({ADAM_ITERS//N_CHUNK} chunks of {N_CHUNK})')
print(f'N_OBS = {N_OBS}, L-BFGS capped at {LBFGS_ITERS} iters')
print(f'Noise sweep: {"ON" if NOISE_SWEEP else "OFF"}, levels = {NOISE_LEVELS}')

## 2. Analytical solution (2D)

Closed-form for an infinite planar domain (see Appendix B):

$$C(x, y, t) = \frac{C_0\,\sigma^2}{\sigma^2 + 2Dt} \exp\!\left(-\frac{x^2 + y^2}{2(\sigma^2 + 2Dt)}\right) e^{-kt}$$

Note the amplitude factor is $\sigma^2/(\sigma^2 + 2Dt)$ in 2D (vs. $\sigma/\sqrt{\sigma^2 + 2Dt}$ in 1D).

In [ ]:
def C_analytical(x, y, t, D=D_TRUE, k=K_TRUE, sigma=SIGMA, C0=C0):
    s2 = sigma ** 2 + 2.0 * D * t
    return C0 * sigma**2 / s2 * np.exp(-(x ** 2 + y ** 2) / (2.0 * s2)) * np.exp(-k * t)

xs = np.linspace(-L/2, L/2, 81)
ys = np.linspace(-L/2, L/2, 81)
Xg, Yg = np.meshgrid(xs, ys, indexing='ij')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, t_snap in zip(axes, [0, 5, 10]):
    C = C_analytical(Xg, Yg, t_snap)
    im = ax.imshow(C.T, extent=[-L/2, L/2, -L/2, L/2], origin='lower', cmap='viridis')
    ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
    ax.set_title(f't = {t_snap} ms')
    plt.colorbar(im, ax=ax)
fig.suptitle('Analytical solution (2D Gaussian spreading + reuptake)')
fig.tight_layout(); plt.show()

## 3. Finite-difference reference solver (2D)

Explicit-time 5-point Laplacian stencil with Neumann (zero-flux) boundaries on all four edges. Stability condition in 2D: $\Delta t \le 1/(2D(\Delta x^{-2} + \Delta y^{-2}))$.

In [ ]:
def fd_reference(D=D_TRUE, k=K_TRUE, L=L, T=T, sigma=SIGMA, C0=C0,
                 nx=81, ny=81, nt=None, dt_store=0.02, verbose=True):
    """Explicit 2D FD solver with zero-flux (Neumann) boundaries.

    Returns (x, y, t_frames, frames). Only every `stride`-th time step is
    stored (frames ~dt_store ms apart), which bounds memory at T = 50 ms;
    observations are interpolated linearly in time between stored frames.
    """
    dx = L / (nx - 1)
    dy = L / (ny - 1)
    if nt is None:
        # 2D stability: dt <= 1 / (2D * (1/dx^2 + 1/dy^2)); 5% safety
        dt_max = 0.95 / (2.0 * D * (1.0/dx**2 + 1.0/dy**2))
        nt = int(np.ceil(T / dt_max)) + 1
    dt = T / (nt - 1)
    assert dt <= 1.0 / (2.0 * D * (1.0/dx**2 + 1.0/dy**2)), 'FD stability violated.'
    stride = max(1, int(round(dt_store / dt)))
    x = np.linspace(-L/2, L/2, nx)
    y = np.linspace(-L/2, L/2, ny)
    Xg, Yg = np.meshgrid(x, y, indexing='ij')
    C = C0 * np.exp(-(Xg**2 + Yg**2) / (2.0 * sigma**2))
    frames, t_frames = [C.copy()], [0.0]
    if verbose:
        print(f'FD: nx={nx}, ny={ny}, nt={nt}, dt={dt:.4e} ms, storing every {stride} steps')
    for n in range(1, nt):
        lap = np.zeros_like(C)
        # d^2/dx^2 with Neumann mirror at x = +-L/2
        lap[1:-1, :] += (C[2:, :] - 2*C[1:-1, :] + C[:-2, :]) / dx**2
        lap[0,   :]  += 2 * (C[1,  :] - C[0,  :]) / dx**2
        lap[-1,  :]  += 2 * (C[-2, :] - C[-1, :]) / dx**2
        # d^2/dy^2 with Neumann mirror at y = +-L/2
        lap[:, 1:-1] += (C[:, 2:] - 2*C[:, 1:-1] + C[:, :-2]) / dy**2
        lap[:, 0]    += 2 * (C[:, 1]  - C[:, 0])  / dy**2
        lap[:, -1]   += 2 * (C[:, -2] - C[:, -1]) / dy**2
        C = C + D * dt * lap - k * dt * C
        if n % stride == 0 or n == nt - 1:
            frames.append(C.copy())
            t_frames.append(n * dt)
    return x, y, np.array(t_frames), np.stack(frames)


# ---- FD reference ("oracle") estimator ------------------------------------
from scipy.optimize import least_squares

def fd_predict(D, k, x_o, y_o, t_o):
    """FD-solver concentrations at the observation points (x_o, y_o, t_o)."""
    x_g, y_g, t_g, H = fd_reference(D=D, k=k, verbose=False)
    interp = RegularGridInterpolator((t_g, x_g, y_g), H,
                                     bounds_error=False, fill_value=0.0)
    return interp(np.stack([t_o, x_o, y_o], axis=1))


def fd_fit(x_o, y_o, t_o, C_o, noise_pct=2.0, D0=None, k0=None):
    """Reference estimator: fit (D, k) with the FD solver itself.

    Nonlinear least squares on (log D, log k) using the same bounded-domain
    FD model that generated the synthetic data, so its error is set only by
    the observation noise: the best any method can do with these data.
    This is a deliberate 'inverse crime' and serves as a benchmark, not a
    competitor; the PINN never sees the FD solver.

    Returns D, k and the Gauss-Newton covariance of (log D, log k), i.e. the
    Laplace approximation of the posterior under a flat prior.
    """
    D0 = D_INIT if D0 is None else D0
    k0 = K_INIT if k0 is None else k0
    resid = lambda th: fd_predict(np.exp(th[0]), np.exp(th[1]), x_o, y_o, t_o) - C_o
    r = least_squares(resid, np.log([D0, k0]), diff_step=1e-3)
    # Noise variance: known for noisy data; residual-based for noise-free data
    s2 = (noise_pct / 100.0 * C0) ** 2 if noise_pct > 0 else float(np.mean(r.fun ** 2))
    cov_log = np.linalg.inv(r.jac.T @ r.jac) * s2
    return {'D': float(np.exp(r.x[0])), 'k': float(np.exp(r.x[1])),
            'cov_log': cov_log, 'nfev': int(r.nfev)}


t0 = time.time()
x_fd, y_fd, t_fd, C_fd_full = fd_reference()
print(f'FD run: {time.time()-t0:.1f} s, shape={C_fd_full.shape}, max C={C_fd_full.max():.4f} mu_M')

## 4. Forward PINN (2D, Flax NNX)

Network input is $(x, y, t) \in \mathbb{R}^3$, fed to the network unscaled (rescaling to $[-1, 1]^3$ was tested and degraded the fit of the fast early-time dynamics). Derivatives of $C$ with respect to each input are taken via `jax.grad`, giving the PDE residual at each collocation point:

$$\mathcal{L}_r(\theta) = \big| \partial_t \hat{C} - D\,(\partial_{xx} + \partial_{yy})\hat{C} + k\,\hat{C} \big|^2.$$

`nnx.jit` JIT-compiles the training step, and `nnx.value_and_grad` fuses forward + backward into one traced call. The Adam phase runs in Optax, and the second-order L-BFGS phase runs in jaxopt — both operating on the same NNX parameter pytree via `nnx.split`/`nnx.merge`.

In [ ]:
# ---- Flax NNX MLP ---------------------------------------------------------
class MLP(nnx.Module):
    """Fully-connected feedforward net with tanh activations.
    Input: (x, y, t) in R^3. Output: scalar C(x, y, t).

    All Linear layers carry param_dtype=float64 so the parameter pytree
    is dtype-homogeneous (matches the float64 D_log/k_log in InverseMLP
    and avoids 'Found more than one dtype in the tree' from jaxopt LBFGS).
    """
    def __init__(self, layers, *, rngs: nnx.Rngs):
        self.n_layers = len(layers) - 1
        for i in range(self.n_layers):
            setattr(self, f'lin_{i}',
                    nnx.Linear(layers[i], layers[i + 1],
                               kernel_init=nnx.initializers.glorot_normal(),
                               bias_init=nnx.initializers.zeros_init(),
                               param_dtype=jnp.float64,
                               rngs=rngs))

    def __call__(self, xyt):
        # Raw (x, y, t) inputs on purpose: rescaling them to [-1, 1]^3 was
        # tested and made the forward fit markedly worse (it compresses the
        # fast early-time dynamics, ~sigma^2 / 2D = 0.4 ms, into a steep
        # feature in scaled time).
        h = xyt
        for i in range(self.n_layers - 1):
            h = jnp.tanh(getattr(self, f'lin_{i}')(h))
        return getattr(self, f'lin_{self.n_layers - 1}')(h)


def C_at(model, x, y, t):
    return model(jnp.stack([x, y, t]))[0]


def pde_residual_one(model, x, y, t, D, k):
    """dC/dt - D Laplacian(C) + k C at one point."""
    dC_dt   = jax.grad(C_at, argnums=3)(model, x, y, t)
    d2C_dx2 = jax.grad(lambda u: jax.grad(C_at, argnums=1)(model, u, y, t))(x)
    d2C_dy2 = jax.grad(lambda v: jax.grad(C_at, argnums=2)(model, x, v, t))(y)
    return dC_dt - D * (d2C_dx2 + d2C_dy2) + k * C_at(model, x, y, t)


def bc_normal_deriv(model, x, y, t, nx, ny):
    dC_dx = jax.grad(C_at, argnums=1)(model, x, y, t)
    dC_dy = jax.grad(C_at, argnums=2)(model, x, y, t)
    return nx * dC_dx + ny * dC_dy


def sample_points(n_domain, n_initial, n_boundary, seed=SEED):
    """Collocation point sampling.

    Interior points (n_domain) are drawn by Latin Hypercube Sampling in
    the 3D (x, y, t) space. Initial and boundary points use uniform
    random sampling.
    """
    rng = np.random.default_rng(seed)

    # Latin Hypercube in 3D (x, y, t) -> [0,1]^3 -> physical domain
    lhs = qmc.LatinHypercube(d=3, seed=seed)
    u = lhs.random(n_domain)
    x_r = -L/2 + L * u[:, 0]
    y_r = -L/2 + L * u[:, 1]
    t_r =        T * u[:, 2]

    # IC points (t = 0): uniform 2D
    x_i = rng.uniform(-L/2, L/2, n_initial)
    y_i = rng.uniform(-L/2, L/2, n_initial)
    C_i = C0 * np.exp(-(x_i**2 + y_i**2) / (2*SIGMA**2))

    # Boundary: split evenly across the four edges (outward unit normals)
    per_edge = n_boundary // 4
    e_left  = (np.full(per_edge, -L/2), rng.uniform(-L/2, L/2, per_edge),
               rng.uniform(0, T, per_edge), np.full(per_edge, -1.0),
               np.full(per_edge,  0.0))
    e_right = (np.full(per_edge,  L/2), rng.uniform(-L/2, L/2, per_edge),
               rng.uniform(0, T, per_edge), np.full(per_edge,  1.0),
               np.full(per_edge,  0.0))
    e_bot   = (rng.uniform(-L/2, L/2, per_edge), np.full(per_edge, -L/2),
               rng.uniform(0, T, per_edge), np.full(per_edge, 0.0),
               np.full(per_edge, -1.0))
    e_top   = (rng.uniform(-L/2, L/2, per_edge), np.full(per_edge,  L/2),
               rng.uniform(0, T, per_edge), np.full(per_edge, 0.0),
               np.full(per_edge,  1.0))
    edges = [e_left, e_right, e_bot, e_top]
    x_b  = np.concatenate([e[0] for e in edges])
    y_b  = np.concatenate([e[1] for e in edges])
    t_b  = np.concatenate([e[2] for e in edges])
    nx_b = np.concatenate([e[3] for e in edges])
    ny_b = np.concatenate([e[4] for e in edges])

    return {k: jnp.asarray(v) for k, v in dict(
        x_r=x_r, y_r=y_r, t_r=t_r,
        x_i=x_i, y_i=y_i, C_i=C_i,
        x_b=x_b, y_b=y_b, t_b=t_b, nx_b=nx_b, ny_b=ny_b,
    ).items()}


def _component_losses(model, pts, D, k):
    res = jax.vmap(pde_residual_one, in_axes=(None, 0, 0, 0, None, None))(
        model, pts['x_r'], pts['y_r'], pts['t_r'], D, k)
    L_r = jnp.mean(res ** 2)

    C_pred_ic = jax.vmap(C_at, in_axes=(None, 0, 0, None))(
        model, pts['x_i'], pts['y_i'], jnp.float64(0.0))
    L_i = jnp.mean((C_pred_ic - pts['C_i']) ** 2)

    nd = jax.vmap(bc_normal_deriv, in_axes=(None, 0, 0, 0, 0, 0))(
        model, pts['x_b'], pts['y_b'], pts['t_b'], pts['nx_b'], pts['ny_b'])
    L_b = jnp.mean(nd ** 2)
    return L_r, L_i, L_b


def forward_loss(model, pts):
    L_r, L_i, L_b = _component_losses(model, pts, D_TRUE, K_TRUE)
    return 10.0 * L_r + L_i + L_b


pts = sample_points(N_DOMAIN, N_INITIAL, N_BOUNDARY)
print(f'Sampled {pts["x_r"].shape[0]} interior (LHS), '
      f'{pts["x_i"].shape[0]} IC, '
      f'{pts["x_b"].shape[0]} boundary points (dtype={pts["x_r"].dtype})')

In [ ]:
# ---- Build model and Optax optimizer --------------------------------------
rngs = nnx.Rngs(SEED)
model_fwd = MLP(LAYERS, rngs=rngs)
optimizer = nnx.Optimizer(model_fwd, optax.adam(LR), wrt=nnx.Param)


# ---- Chunked Adam: N_CHUNK steps per JIT'd call ---------------------------
@nnx.jit(static_argnames=('n_chunk',))
def adam_chunk(model, optimizer, pts, n_chunk):
    last_loss = jnp.zeros(())
    for _ in range(n_chunk):
        loss, grads = nnx.value_and_grad(forward_loss)(model, pts)
        optimizer.update(model, grads)
        last_loss = loss
    return last_loss


# ---- Adam phase -----------------------------------------------------------
print(f'Forward Adam: {ADAM_ITERS} iters in chunks of {N_CHUNK}')
print(f'(first chunk includes JIT compile -- expect 30s-2min on Colab T4)')
t0 = time.time()
n_chunks = ADAM_ITERS // N_CHUNK
for c in range(n_chunks):
    loss_val = adam_chunk(model_fwd, optimizer, pts, N_CHUNK)
    if c == 0 or (c + 1) % 20 == 0 or c == n_chunks - 1:
        elapsed = time.time() - t0
        steps_done = (c + 1) * N_CHUNK
        print(f'  chunk {c+1:>4d}/{n_chunks}  step {steps_done:>5d}  '
              f'loss = {float(loss_val):.4e}  elapsed = {elapsed:.1f}s')
print(f'Adam phase total: {time.time() - t0:.1f} s')


# ---- L-BFGS phase via jaxopt ----------------------------------------------
# Cast the parameter pytree to float64 before handing it to jaxopt.
# Even with param_dtype=float64 set everywhere, NNX bookkeeping arrays
# can sneak in as float32; jaxopt LBFGS rejects mixed-dtype pytrees.
print(f'\nForward L-BFGS: capped at {LBFGS_ITERS} iters')
gdef, state = nnx.split(model_fwd)
state = jax.tree.map(
    lambda x: x.astype(jnp.float64) if hasattr(x, 'dtype') and jnp.issubdtype(x.dtype, jnp.floating) else x,
    state,
)

def lbfgs_loss(params):
    return forward_loss(nnx.merge(gdef, params), pts)

t0 = time.time()
solver = LBFGS(fun=lbfgs_loss, maxiter=LBFGS_ITERS, tol=1e-9)
result = solver.run(state)
model_fwd = nnx.merge(gdef, result.params)
print(f'L-BFGS phase: {time.time() - t0:.1f} s')
print(f'Final L-BFGS loss: {float(result.state.value):.4e}')

### 4.1 Forward-problem evaluation

Evaluate the PINN on a 3D space-time grid; compute L2 errors vs. analytical and FD references; report per-snapshot errors.

In [ ]:
# Evaluation grid: 41 x 41 spatial x (T + 1) time slices (1 ms spacing)
nx_eval, ny_eval, nt_eval = 41, 41, int(T) + 1
xs = np.linspace(-L/2, L/2, nx_eval)
ys = np.linspace(-L/2, L/2, ny_eval)
ts = np.linspace(0, T, nt_eval)
Xg, Yg, Tg = np.meshgrid(xs, ys, ts, indexing='ij')
XYT = np.stack([Xg.ravel(), Yg.ravel(), Tg.ravel()], axis=1)

# JIT-batched prediction over the full grid
@jax.jit
def predict_batch(model_state, xyt_batch):
    m = nnx.merge(gdef, model_state)
    return jax.vmap(lambda row: m(row)[0])(xyt_batch)

# Predict in chunks of 50k points to avoid OOM on T4
_, state_fwd = nnx.split(model_fwd)
C_flat = []
chunk = 50_000
for i in range(0, XYT.shape[0], chunk):
    batch = jnp.asarray(XYT[i:i+chunk])
    C_flat.append(np.asarray(predict_batch(state_fwd, batch)))
C_pinn = np.concatenate(C_flat).reshape(nx_eval, ny_eval, nt_eval)

# Analytical and FD references on the same grid
C_exact = C_analytical(Xg, Yg, Tg)
interp  = RegularGridInterpolator((t_fd, x_fd, y_fd), C_fd_full,
                                  bounds_error=False, fill_value=0.0)
C_fd    = interp(np.stack([Tg.ravel(), Xg.ravel(), Yg.ravel()], axis=1)
                ).reshape(nx_eval, ny_eval, nt_eval)

err_anal = 100.0 * np.linalg.norm(C_pinn - C_exact) / np.linalg.norm(C_exact)
err_fd   = 100.0 * np.linalg.norm(C_pinn - C_fd) / np.linalg.norm(C_fd)
print(f'L2(PINN vs. analytical, full window) = {err_anal:.3f}%')
print(f'L2(PINN vs. FD reference, full window) = {err_fd:.3f}%')

per_snapshot = {}
for t_snap in (1.0, 5.0, 10.0, 20.0, 50.0):
    i_t = np.argmin(np.abs(ts - t_snap))
    C_p = C_pinn[:, :, i_t]
    C_a = C_exact[:, :, i_t]
    C_f = C_fd[:, :, i_t]
    per_snapshot[f't_{int(t_snap)}'] = {
        'L2_anal_pct': 100.0 * np.linalg.norm(C_p - C_a) / np.linalg.norm(C_a),
        'L2_fd_pct':   100.0 * np.linalg.norm(C_p - C_f) / np.linalg.norm(C_f),
    }
    e = per_snapshot[f't_{int(t_snap)}']
    print(f'  t = {t_snap:5.1f} ms:  vs. analytical = {e["L2_anal_pct"]:6.2f}%   vs. FD = {e["L2_fd_pct"]:6.2f}%')

In [ ]:
# Snapshot panel: heatmaps of C(x, y) at three time slices, all showing the PINN prediction
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = C_pinn.max()
for ax, t_snap in zip(axes, [0.0, T/4, T/2]):
    i_t = np.argmin(np.abs(ts - t_snap))
    im = ax.imshow(C_pinn[:, :, i_t].T, extent=[-L/2, L/2, -L/2, L/2],
                   origin='lower', cmap='viridis', vmin=0, vmax=vmax)
    ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
    ax.set_title(f't = {t_snap:.1f} ms')
    plt.colorbar(im, ax=ax, label='C (mu_M)')
fig.suptitle('Forward problem: PINN concentration field C(x, y, t)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'forward_snapshots.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Comparison panel at t = T/4: analytical, PINN, |error|
i_t = np.argmin(np.abs(ts - T/4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, field, title in zip(
    axes,
    [C_exact[:, :, i_t], C_pinn[:, :, i_t], np.abs(C_exact[:, :, i_t] - C_pinn[:, :, i_t])],
    ['Analytical', 'PINN', '|Analytical - PINN|'],
):
    im = ax.imshow(field.T, extent=[-L/2, L/2, -L/2, L/2],
                   origin='lower', cmap='viridis')
    ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
fig.suptitle(f'C(x, y) at t = {T/4:.1f} ms')
fig.tight_layout()
fig.savefig(FIG_DIR / 'forward_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Inverse PINN — recover $D$ and $k$ from noisy observations

400 observations sampled uniformly over $(x, y) \in \Omega$ and $t \in [0.5, 0.9T]$, with 2% Gaussian noise. Clean concentrations come from the FD reference (consistent with the PINN's Neumann BCs). Trainable parameters are log-parameterized to enforce positivity, and the residual loss is up-weighted $10\times$ vs. the data loss.

**Multi-seed evaluation.** To quantify run-to-run variability (rather than reporting a single anecdotal recovery), we re-train the inverse PINN over **5 different network-initialization seeds** with the *same* synthetic observation data. The final manuscript table reports mean ± std across seeds.

Runtime on Colab T4: ~60–90 min for 5 seeds (first seed pays the JIT compile cost; subsequent seeds run faster on the cached graph). Reduce `INVERSE_SEEDS` to 3 if you want to save time.

In [ ]:
def make_noisy_observations(n_obs=N_OBS, noise_pct=2.0, rng=None):
    """Sample noisy observations from the FD reference.

    Time window is [0.5, 0.9*T] = [0.5, 45] ms at T = 50 ms, i.e. about
    one uptake time constant (1/k = 50 ms at k = 0.020 1/ms). A 20 ms
    window contains too little decay to identify the reuptake rate k.
    """
    rng = rng or np.random.default_rng(SEED)
    obs_x = rng.uniform(-L/2, L/2, n_obs)
    obs_y = rng.uniform(-L/2, L/2, n_obs)
    obs_t = rng.uniform(0.5, 0.9 * T, n_obs)
    interp_fd = RegularGridInterpolator((t_fd, x_fd, y_fd), C_fd_full,
                                        bounds_error=False, fill_value=0.0)
    C_clean = interp_fd(np.stack([obs_t, obs_x, obs_y], axis=1))
    obs_C = C_clean + rng.normal(0.0, (noise_pct / 100.0) * C0, n_obs)
    return obs_x, obs_y, obs_t, obs_C

obs_x, obs_y, obs_t, obs_C = make_noisy_observations(n_obs=N_OBS, noise_pct=2.0)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(obs_x, obs_y, c=obs_t, cmap='plasma', s=20, alpha=0.7)
ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
ax.set_xlim(-L/2, L/2); ax.set_ylim(-L/2, L/2)
ax.set_title(f'{len(obs_x)} noisy observations '
             f'(color = t in [0.5, {0.9*T:.0f}] ms, 2% noise)')
plt.colorbar(sc, ax=ax, label='t (ms)')
plt.show()

In [ ]:
# ---- Inverse model: MLP + log-parametrized D, k ---------------------------
class InverseMLP(nnx.Module):
    """MLP plus log-parametrized D and k as learnable scalars (float64)."""
    def __init__(self, layers, D0, k0, *, rngs: nnx.Rngs):
        self.mlp = MLP(layers, rngs=rngs)
        self.D_log = nnx.Param(jnp.asarray(np.log(D0), dtype=jnp.float64))
        self.k_log = nnx.Param(jnp.asarray(np.log(k0), dtype=jnp.float64))

    def __call__(self, xyt):
        return self.mlp(xyt)

    @property
    def D(self):
        getter = getattr(self.D_log, 'get_value', None)
        return jnp.exp(getter() if callable(getter) else self.D_log.value)

    @property
    def k(self):
        getter = getattr(self.k_log, 'get_value', None)
        return jnp.exp(getter() if callable(getter) else self.k_log.value)


def inverse_loss(model, pts, obs):
    D, k = model.D, model.k
    L_r, L_i, L_b = _component_losses(model.mlp, pts, D, k)
    C_pred_obs = jax.vmap(C_at, in_axes=(None, 0, 0, 0))(
        model.mlp, obs['x_d'], obs['y_d'], obs['t_d'])
    L_d = jnp.mean((C_pred_obs - obs['C_d']) ** 2)
    return 10.0 * L_r + L_i + L_b + L_d


# ---- Build, train, and recover (per-seed) ---------------------------------
def train_inverse_once(obs, seed, write_variables_dat=False, return_model=False):
    """Run one full inverse-training pass with the given network-init seed.

    Returns (D_rec, k_rec, history) by default; if return_model=True, also
    returns the converged InverseMLP (needed by the Laplace UQ cell).
    """
    rngs = nnx.Rngs(seed)
    model = InverseMLP(LAYERS, D0=D_INIT, k0=K_INIT, rngs=rngs)
    optimizer = nnx.Optimizer(model, optax.adam(LR), wrt=nnx.Param)

    history = []

    @nnx.jit(static_argnames=('n_chunk',))
    def adam_chunk_inv(model, optimizer, pts, obs, n_chunk):
        last_loss = jnp.zeros(())
        for _ in range(n_chunk):
            loss, grads = nnx.value_and_grad(inverse_loss)(model, pts, obs)
            optimizer.update(model, grads)
            last_loss = loss
        return last_loss

    t0 = time.time()
    n_chunks = ADAM_ITERS // N_CHUNK
    for c in range(n_chunks):
        loss_val = adam_chunk_inv(model, optimizer, pts, obs, N_CHUNK)
        steps_done = (c + 1) * N_CHUNK
        if steps_done % 500 == 0:
            history.append((steps_done, float(model.D), float(model.k)))
        if c == 0 or (c + 1) % 40 == 0 or c == n_chunks - 1:
            elapsed = time.time() - t0
            print(f'    [seed {seed}] chunk {c+1:>4d}/{n_chunks}  '
                  f'step {steps_done:>5d}  loss = {float(loss_val):.4e}  '
                  f'D = {float(model.D):.4f}  k = {float(model.k):.4f}  '
                  f'elapsed = {elapsed:.1f}s')
    adam_time = time.time() - t0

    # ---- L-BFGS phase ----
    gdef_inv, state_inv = nnx.split(model)
    state_inv = jax.tree.map(
        lambda x: x.astype(jnp.float64) if hasattr(x, 'dtype') and jnp.issubdtype(x.dtype, jnp.floating) else x,
        state_inv,
    )

    def lbfgs_loss_inv(params):
        m = nnx.merge(gdef_inv, params)
        return inverse_loss(m, pts, obs)

    t0 = time.time()
    solver = LBFGS(fun=lbfgs_loss_inv, maxiter=LBFGS_ITERS, tol=1e-9)
    result = solver.run(state_inv)
    model = nnx.merge(gdef_inv, result.params)
    lbfgs_time = time.time() - t0

    D_rec, k_rec = float(model.D), float(model.k)
    history.append((ADAM_ITERS + LBFGS_ITERS, D_rec, k_rec))
    print(f'    [seed {seed}] Adam={adam_time:.1f}s, LBFGS={lbfgs_time:.1f}s, '
          f'D = {D_rec:.4f}, k = {k_rec:.4f}')

    if write_variables_dat:
        with open(FIG_DIR / 'variables.dat', 'w') as fh:
            for it, D, k in history:
                fh.write(f'{it}\t[{np.log(D):.6f}, {np.log(k):.6f}]\n')

    if return_model:
        return D_rec, k_rec, history, model
    return D_rec, k_rec, history


# ---- Multi-seed driver ----------------------------------------------------
INVERSE_SEEDS = [1234, 1235, 1236, 1237, 1238]

obs = {key: jnp.asarray(np.asarray(v, dtype=np.float64))
       for key, v in dict(x_d=obs_x, y_d=obs_y, t_d=obs_t, C_d=obs_C).items()}

print('=' * 72)
print(f'  Multi-seed inverse PINN over {len(INVERSE_SEEDS)} seeds: {INVERSE_SEEDS}')
print('=' * 72)

per_seed_D, per_seed_k = [], []
per_seed_histories = {}
model_inv_primary = None      # Save the first seed's converged model for the UQ cell
overall_t0 = time.time()

for i, seed in enumerate(INVERSE_SEEDS):
    print(f'\n--- Seed {seed} ({i+1}/{len(INVERSE_SEEDS)}) ---')
    write_vars = (i == 0)
    if i == 0:
        D_rec, k_rec, history, model_inv_primary = train_inverse_once(
            obs, seed, write_variables_dat=write_vars, return_model=True)
    else:
        D_rec, k_rec, history = train_inverse_once(
            obs, seed, write_variables_dat=write_vars)
    per_seed_D.append(D_rec)
    per_seed_k.append(k_rec)
    per_seed_histories[seed] = history
    print(f'    [seed {seed}] running elapsed: {(time.time() - overall_t0)/60:.1f} min')

print('\n' + '=' * 72)
print(f'  All {len(INVERSE_SEEDS)} seeds complete in {(time.time() - overall_t0)/60:.1f} min')
print('=' * 72)

# ---- Aggregate stats -----------------------------------------------------
per_seed_D = np.array(per_seed_D)
per_seed_k = np.array(per_seed_k)
D_mean, D_std = float(per_seed_D.mean()), float(per_seed_D.std(ddof=1))
k_mean, k_std = float(per_seed_k.mean()), float(per_seed_k.std(ddof=1))

per_seed_D_err_pct = 100.0 * np.abs(per_seed_D - D_TRUE) / D_TRUE
per_seed_k_err_pct = 100.0 * np.abs(per_seed_k - K_TRUE) / K_TRUE
D_err_mean, D_err_std = float(per_seed_D_err_pct.mean()), float(per_seed_D_err_pct.std(ddof=1))
k_err_mean, k_err_std = float(per_seed_k_err_pct.mean()), float(per_seed_k_err_pct.std(ddof=1))

print()
print(f'  Per-seed D:   {per_seed_D}')
print(f'  Per-seed k:   {per_seed_k}')
print()
print(f'  D = {D_mean:.4f} +/- {D_std:.4f}  mu_m^2/ms   (true {D_TRUE})')
print(f'    |err| = {D_err_mean:.2f}% +/- {D_err_std:.2f}%')
print(f'  k = {k_mean:.4f} +/- {k_std:.4f}  1/ms        (true {K_TRUE})')
print(f'    |err| = {k_err_mean:.2f}% +/- {k_err_std:.2f}%')

# Expose primary-seed values for downstream cells
D_rec, k_rec = per_seed_D[0], per_seed_k[0]
rel_D, rel_k = float(per_seed_D_err_pct[0]), float(per_seed_k_err_pct[0])

In [ ]:
# Convergence plot: parse DeepXDE's bracketed variables.dat and apply exp() to recover D, k
iters, Ds, ks = [], [], []
with open(FIG_DIR / 'variables.dat') as fh:
    for line in fh:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        try:
            it = int(parts[0])
        except ValueError:
            continue
        nums = re.findall(r'[+-]?\d+\.?\d*(?:[eE][+-]?\d+)?', parts[1])
        if len(nums) >= 2:
            iters.append(it)
            Ds.append(np.exp(float(nums[0])))
            ks.append(np.exp(float(nums[1])))
iters, Ds, ks = np.array(iters), np.array(Ds), np.array(ks)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(iters, Ds, 'r-')
axes[0].axhline(D_TRUE, color='k', ls=':', label='true')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('D (mu_m^2/ms)')
axes[0].set_title('Diffusion coefficient recovery'); axes[0].legend()
axes[1].plot(iters, ks, 'b-')
axes[1].axhline(K_TRUE, color='k', ls=':', label='true')
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('k (1/ms)')
axes[1].set_title('Reuptake rate recovery'); axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'inverse_convergence.png', dpi=200, bbox_inches='tight')
plt.show()

## 5.4 Posterior Uncertainty (Laplace Approximation, FD Likelihood) and Reference Estimator

**Reference ("oracle") estimator.** We fit $(D, k)$ to the same 400 observations by nonlinear least squares using the bounded-domain FD solver itself as the forward model. Because this is the model that generated the data, its error reflects the observation noise alone: it is the best any method can achieve on these data and serves as the benchmark for the PINN (a deliberate "inverse crime"; the PINN never sees the FD solver).

**Laplace approximation.** Around the reference MAP we approximate the posterior as a Gaussian, with covariance equal to the inverse Gauss–Newton Hessian of the Gaussian negative log-likelihood (flat prior on $\log D$, $\log k$):

$$p(D, k \mid \text{data}) \approx \mathcal{N}\big((\hat D, \hat k),\; H^{-1}\big),\quad H = J^\top J / \sigma_\eta^2,$$

with $J$ the Jacobian of the FD predictions with respect to $(\log D, \log k)$, mapped to $(D, k)$ by the delta method.

*Why the FD likelihood:* an earlier version used the infinite-domain analytical Gaussian as the likelihood. That model has no wall reflections, so it drove $k \to 0$ and biased $D$; the FD likelihood has the correct bounded-domain physics.

Runtime: ~1-3 min (a few dozen FD solves).

In [ ]:
# ---- Posterior over (D, k): Laplace approximation, FD likelihood ---------
# The likelihood uses the bounded-domain FD solver (the same physics as the
# data), not the infinite-domain analytical Gaussian: the analytical model
# has no wall reflections, so it drives k to 0 and biases D. The Laplace
# covariance is the inverse Gauss-Newton Hessian at the MAP (flat prior on
# log D, log k), mapped to (D, k) by the delta method.

def _read_log_param(p):
    g = getattr(p, 'get_value', None)
    return g() if callable(g) else p.value

D_pinn = float(np.exp(_read_log_param(model_inv_primary.D_log)))
k_pinn = float(np.exp(_read_log_param(model_inv_primary.k_log)))
print(f'PINN (primary seed):         D = {D_pinn:.4f}, k = {k_pinn:.5f}')

t0 = time.time()
fd_ref = fd_fit(obs_x, obs_y, obs_t, obs_C, noise_pct=2.0)
fd_ref_time = time.time() - t0
D_star, k_star = fd_ref['D'], fd_ref['k']
print(f'FD reference MAP (oracle):   D = {D_star:.4f}, k = {k_star:.5f}  '
      f'({fd_ref["nfev"]} FD solves, {fd_ref_time:.1f} s)')

# ---- Delta-method transform to (D, k) space ------------------------------
Jd = np.diag([D_star, k_star])
Sigma_Dk = Jd @ fd_ref['cov_log'] @ Jd
D_post_std = float(np.sqrt(Sigma_Dk[0, 0]))
k_post_std = float(np.sqrt(Sigma_Dk[1, 1]))
rho_Dk     = float(Sigma_Dk[0, 1] / (D_post_std * k_post_std))

D_lo, D_hi = D_star - 1.96 * D_post_std, D_star + 1.96 * D_post_std
k_lo, k_hi = k_star - 1.96 * k_post_std, k_star + 1.96 * k_post_std
D_covers = D_lo <= D_TRUE <= D_hi
k_covers = k_lo <= K_TRUE <= k_hi

D_ref_err = 100.0 * abs(D_star - D_TRUE) / D_TRUE
k_ref_err = 100.0 * abs(k_star - K_TRUE) / K_TRUE

print('\n=== Posterior over (D, k): Laplace, FD likelihood ===')
print(f'  D = {D_star:.4f} +/- {D_post_std:.4f}  (95% CI: [{D_lo:.4f}, {D_hi:.4f}])')
print(f'  k = {k_star:.5f} +/- {k_post_std:.5f}  (95% CI: [{k_lo:.5f}, {k_hi:.5f}])')
print(f'  Correlation rho(D, k) = {rho_Dk:+.4f}')
print(f'  Truth coverage: D {"YES" if D_covers else "NO"}, k {"YES" if k_covers else "NO"}')

print('\n=== PINN vs reference (oracle) on the same 2%-noise data ===')
print(f'                       D err       k err')
print(f'  Reference (FD):   {D_ref_err:>8.2f}%  {k_ref_err:>8.2f}%')
print(f'  PINN (5 seeds):   {D_err_mean:>8.2f}%  {k_err_mean:>8.2f}%')

# ---- Sample from the Gaussian posterior and plot ------------------------
n_post_samples = 2000
rng_post = np.random.default_rng(SEED)
post_samples = rng_post.multivariate_normal(
    mean=[D_star, k_star], cov=Sigma_Dk, size=n_post_samples)

from matplotlib.patches import Ellipse

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(post_samples[:, 0], post_samples[:, 1],
           s=3, alpha=0.25, color='C0',
           label=f'{n_post_samples} Laplace posterior samples')

eigvals, eigvecs = np.linalg.eigh(Sigma_Dk)
angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
for nsig, ls in zip([1, 2, 3], ['-', '--', ':']):
    width  = 2 * nsig * np.sqrt(eigvals[1])
    height = 2 * nsig * np.sqrt(eigvals[0])
    ell = Ellipse(xy=(D_star, k_star), width=width, height=height,
                  angle=angle, edgecolor='navy', facecolor='none',
                  lw=1.5, linestyle=ls,
                  label=f'Laplace {nsig}sigma' if nsig == 1 else None)
    ax.add_patch(ell)

ax.plot(D_star, k_star, 'r*', markersize=20,
        label=f'FD reference MAP (D={D_star:.3f}, k={k_star:.4f})')
ax.plot(per_seed_D, per_seed_k, 'gs', markersize=8,
        label='PINN, 5 seeds')
ax.plot(D_TRUE, K_TRUE, 'k+', markersize=18, mew=3,
        label=f'Truth (D={D_TRUE}, k={K_TRUE})')

ax.set_xlabel(r'Diffusion coefficient $D$ ($\mu$m$^2$/ms)')
ax.set_ylabel(r'Reuptake rate $k$ (1/ms)')
ax.set_title(f'Joint posterior $p(D, k\\,|\\,\\mathrm{{data}})$ '
             f'-- Laplace (FD likelihood)\n'
             f'$\\rho(D, k) = {rho_Dk:+.3f}$')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'posterior.png', dpi=200, bbox_inches='tight')
print(f'\nSaved {FIG_DIR / "posterior.png"}')
plt.show()

## 5.5 Noise Sensitivity Sweep

How does inverse recovery degrade as observational noise increases? We re-run the inverse PINN at five noise levels $\eta \in \{0\%, 1\%, 2\%, 5\%, 10\%\}$ — from clean data through the realistic FSCV regime (~2%) up to a stress-test level — using a single seed (1234) at each noise level so that the recovery variability reflects noise alone, not network initialization. Each level is also fitted with the FD reference estimator, giving the noise-limited floor for comparison.

Runtime: ~40-60 min for the 5-level sweep on Colab T4 (first noise level pays JIT compile cost; subsequent levels run faster on the cached graph). Skip the entire block by setting `NOISE_SWEEP = False` in Cell 4.

In [ ]:
# Initialise sweep storage so downstream cells run cleanly even if the
# sweep is skipped (NOISE_SWEEP = False).
sweep_D = np.array([])
sweep_k = np.array([])
sweep_D_err = np.array([])
sweep_k_err = np.array([])
sweep_D_ref = np.array([])
sweep_k_ref = np.array([])
sweep_D_ref_err = np.array([])
sweep_k_ref_err = np.array([])

if NOISE_SWEEP:
    print('=' * 72)
    print(f'  Noise sensitivity sweep over {len(NOISE_LEVELS)} levels: {NOISE_LEVELS}')
    print(f'  (single seed = {SEED} at each level; isolates noise from init)')
    print('=' * 72)

    sweep_D_list, sweep_k_list = [], []
    sweep_D_ref_list, sweep_k_ref_list = [], []
    sweep_t0 = time.time()
    for noise in NOISE_LEVELS:
        print(f'\n--- Noise = {noise}% ---')
        obs_x_n, obs_y_n, obs_t_n, obs_C_n = make_noisy_observations(
            n_obs=N_OBS, noise_pct=noise,
            rng=np.random.default_rng(SEED),
        )
        obs_n = {key: jnp.asarray(np.asarray(v, dtype=np.float64))
                 for key, v in dict(x_d=obs_x_n, y_d=obs_y_n,
                                    t_d=obs_t_n, C_d=obs_C_n).items()}
        D_n, k_n, _ = train_inverse_once(obs_n, seed=SEED,
                                         write_variables_dat=False)
        ref_n = fd_fit(obs_x_n, obs_y_n, obs_t_n, obs_C_n, noise_pct=noise)
        sweep_D_list.append(D_n)
        sweep_k_list.append(k_n)
        sweep_D_ref_list.append(ref_n['D'])
        sweep_k_ref_list.append(ref_n['k'])
        elapsed = (time.time() - sweep_t0) / 60.0
        print(f'  Noise {noise:>5.1f}%: PINN D = {D_n:.4f}, k = {k_n:.5f} | '
              f'FD ref D = {ref_n["D"]:.4f}, k = {ref_n["k"]:.5f}, '
              f'sweep elapsed = {elapsed:.1f} min')

    sweep_D = np.array(sweep_D_list)
    sweep_k = np.array(sweep_k_list)
    sweep_D_err = 100.0 * np.abs(sweep_D - D_TRUE) / D_TRUE
    sweep_k_err = 100.0 * np.abs(sweep_k - K_TRUE) / K_TRUE
    sweep_D_ref = np.array(sweep_D_ref_list)
    sweep_k_ref = np.array(sweep_k_ref_list)
    sweep_D_ref_err = 100.0 * np.abs(sweep_D_ref - D_TRUE) / D_TRUE
    sweep_k_ref_err = 100.0 * np.abs(sweep_k_ref - K_TRUE) / K_TRUE

    print('\n' + '=' * 72)
    print(f'  Noise Sweep Results (seed {SEED}, {len(NOISE_LEVELS)} levels)')
    print('=' * 72)
    print(f'  {"Noise":>6}  {"D":>8}  {"|D err|%":>9}  {"ref %":>7}  '
          f'{"k":>8}  {"|k err|%":>9}  {"ref %":>7}')
    for i, noise in enumerate(NOISE_LEVELS):
        print(f'  {noise:>5.1f}%  {sweep_D[i]:>8.4f}  {sweep_D_err[i]:>8.2f}%  '
              f'{sweep_D_ref_err[i]:>6.2f}%  {sweep_k[i]:>8.5f}  '
              f'{sweep_k_err[i]:>8.2f}%  {sweep_k_ref_err[i]:>6.2f}%')

    # ---- Plot: recovered params vs noise -----------------------------
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(NOISE_LEVELS, sweep_D, 'ro-', markersize=7, label='PINN')
    axes[0].plot(NOISE_LEVELS, sweep_D_ref, 'k^--', markersize=7, mfc='none',
                 label='FD reference (oracle)')
    axes[0].axhline(D_TRUE, color='k', ls=':', label=f'True ({D_TRUE})')
    axes[0].set_xlabel('Observational noise level (% of $C_0$)')
    axes[0].set_ylabel(r'$D$ ($\mu$m$^2$/ms)')
    axes[0].set_title('Diffusion coefficient recovery vs. noise')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(NOISE_LEVELS, sweep_k, 'bo-', markersize=7, label='PINN')
    axes[1].plot(NOISE_LEVELS, sweep_k_ref, 'k^--', markersize=7, mfc='none',
                 label='FD reference (oracle)')
    axes[1].axhline(K_TRUE, color='k', ls=':', label=f'True ({K_TRUE})')
    axes[1].set_xlabel('Observational noise level (% of $C_0$)')
    axes[1].set_ylabel(r'$k$ (1/ms)')
    axes[1].set_title('Reuptake rate recovery vs. noise')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(FIG_DIR / 'noise_sweep.png', dpi=200, bbox_inches='tight')
    print(f'\nSaved {FIG_DIR / "noise_sweep.png"}')
    plt.show()
else:
    print('NOISE_SWEEP = False, skipping noise sensitivity analysis.')

## 5.6 Parameter Recovery Across a (D, k) Grid

The canonical inverse-problem results so far use a single ground-truth pair $(D^*, k^*) = (0.32, 0.020)$ (Cragg & Rice 2004). To show that the recovery is not tuned to that specific point, we re-run the inverse PINN at five (true) parameter combinations obtained by scaling the canonical values by roughly $0.5\times$ and $2\times$: $D \in \{0.16, 0.32, 0.50\}\,\mu\text{m}^2/\text{ms}$ and $k \in \{0.010, 0.020, 0.040\}\,\text{ms}^{-1}$.

Each combination uses a freshly-generated set of 400 noisy synthetic observations from a finite-difference reference seeded with that specific $(D^*, k^*)$, and the inverse PINN is trained with the canonical hyperparameters (Adam 20k iters + L-BFGS, single seed 1234) starting from the same offset initial guesses $(D_0, k_0) = (0.30, 0.016)$. Each combination is also fitted with the FD reference estimator.

Runtime: ~25 min on A100 (5 combos $\times$ ~5 min each).

In [ ]:
# ---- Parameter recovery across a (D, k) grid ----------------------------
PARAM_GRID = [
    (0.16, 0.010),  # low D, low k
    (0.16, 0.040),  # low D, high k
    (0.32, 0.020),  # canonical (Cragg & Rice 2004)
    (0.50, 0.010),  # high D, low k
    (0.50, 0.040),  # high D, high k
]

def synth_obs_for(D_g, k_g, n_obs=N_OBS, noise_pct=2.0, seed=SEED):
    # Regenerate FD reference at (D_g, k_g) and draw noisy observations.
    rng = np.random.default_rng(seed)
    x_fd_g, y_fd_g, t_fd_g, C_fd_g = fd_reference(D=D_g, k=k_g)
    interp = RegularGridInterpolator((t_fd_g, x_fd_g, y_fd_g), C_fd_g,
                                     bounds_error=False, fill_value=0.0)
    ox = rng.uniform(-L/2, L/2, n_obs)
    oy = rng.uniform(-L/2, L/2, n_obs)
    ot = rng.uniform(0.5, 0.9 * T, n_obs)
    C_clean = interp(np.stack([ot, ox, oy], axis=1))
    C_obs = C_clean + rng.normal(0.0, (noise_pct / 100.0) * C0, n_obs)
    return ox, oy, ot, C_obs


print('=' * 72)
print(f'  Parameter recovery grid over {len(PARAM_GRID)} (D, k) combinations')
print('=' * 72)

param_grid_results = []
pg_t0 = time.time()

for i, (D_g, k_g) in enumerate(PARAM_GRID):
    print(f'\n--- Combination {i+1}/{len(PARAM_GRID)}: (D, k) = ({D_g}, {k_g}) ---')
    ox, oy, ot, oC = synth_obs_for(D_g, k_g, n_obs=N_OBS, noise_pct=2.0, seed=SEED)
    obs_g = {key: jnp.asarray(np.asarray(v, dtype=np.float64))
             for key, v in dict(x_d=ox, y_d=oy, t_d=ot, C_d=oC).items()}
    D_g_rec, k_g_rec, _ = train_inverse_once(obs_g, seed=SEED, write_variables_dat=False)
    ref_g = fd_fit(ox, oy, ot, oC, noise_pct=2.0)
    D_g_err = 100.0 * abs(D_g_rec - D_g) / D_g
    k_g_err = 100.0 * abs(k_g_rec - k_g) / k_g
    param_grid_results.append({
        'D_true': D_g, 'k_true': k_g, 'kT': k_g * T,
        'D_rec':  D_g_rec, 'k_rec':  k_g_rec,
        'D_rel_err_pct': D_g_err, 'k_rel_err_pct': k_g_err,
        'D_ref': ref_g['D'], 'k_ref': ref_g['k'],
        'D_ref_rel_err_pct': 100.0 * abs(ref_g['D'] - D_g) / D_g,
        'k_ref_rel_err_pct': 100.0 * abs(ref_g['k'] - k_g) / k_g,
    })
    elapsed = (time.time() - pg_t0) / 60.0
    print(f'  -> PINN (D, k) = ({D_g_rec:.4f}, {k_g_rec:.5f})  |err| = ({D_g_err:.2f}%, {k_g_err:.2f}%)')
    print(f'  -> FD ref (D, k) = ({ref_g["D"]:.4f}, {ref_g["k"]:.5f})  elapsed = {elapsed:.1f} min')

print('\n' + '=' * 72)
print('  Parameter Recovery Grid (PINN | FD reference)')
print('=' * 72)
print(f'  {"D_true":>7} {"k_true":>7} {"kT":>5} {"D_rec":>8} {"|D err|%":>9} {"ref%":>6} '
      f'{"k_rec":>8} {"|k err|%":>9} {"ref%":>6}')
for r in param_grid_results:
    print(f'  {r["D_true"]:>7.3f} {r["k_true"]:>7.3f} {r["kT"]:>5.2f} {r["D_rec"]:>8.4f} '
          f'{r["D_rel_err_pct"]:>8.2f}% {r["D_ref_rel_err_pct"]:>5.2f}% '
          f'{r["k_rec"]:>8.5f} {r["k_rel_err_pct"]:>8.2f}% {r["k_ref_rel_err_pct"]:>5.2f}%')

# Plot true vs recovered for D and k
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
D_trues = [r['D_true'] for r in param_grid_results]
D_recs  = [r['D_rec']  for r in param_grid_results]
D_refs  = [r['D_ref']  for r in param_grid_results]
k_trues = [r['k_true'] for r in param_grid_results]
k_recs  = [r['k_rec']  for r in param_grid_results]
k_refs  = [r['k_ref']  for r in param_grid_results]

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='y = x (perfect recovery)')
axes[0].scatter(D_trues, D_recs, s=80, color='red', zorder=5, label='PINN')
axes[0].scatter(D_trues, D_refs, s=80, marker='^', facecolors='none',
                edgecolors='k', zorder=6, label='FD reference (oracle)')
for r in param_grid_results:
    axes[0].annotate(f'k={r["k_true"]}', (r['D_true'], r['D_rec']),
                     textcoords='offset points', xytext=(7, 5), fontsize=8)
axes[0].set_xlabel(r'True $D$ ($\mu$m$^2$/ms)')
axes[0].set_ylabel(r'Recovered $D$ ($\mu$m$^2$/ms)')
axes[0].set_title('Diffusion coefficient recovery across parameter grid')
axes[0].set_xlim(0.1, 0.6); axes[0].set_ylim(0.1, 0.6)
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([0, 0.05], [0, 0.05], 'k--', alpha=0.5, label='y = x (perfect recovery)')
axes[1].scatter(k_trues, k_recs, s=80, color='blue', zorder=5, label='PINN')
axes[1].scatter(k_trues, k_refs, s=80, marker='^', facecolors='none',
                edgecolors='k', zorder=6, label='FD reference (oracle)')
for r in param_grid_results:
    axes[1].annotate(f'D={r["D_true"]}', (r['k_true'], r['k_rec']),
                     textcoords='offset points', xytext=(7, 5), fontsize=8)
axes[1].set_xlabel(r'True $k$ (1/ms)')
axes[1].set_ylabel(r'Recovered $k$ (1/ms)')
axes[1].set_title('Reuptake rate recovery across parameter grid')
axes[1].set_xlim(0, 0.05); axes[1].set_ylim(0, 0.05)
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'param_grid.png', dpi=200, bbox_inches='tight')
print(f'\nSaved {FIG_DIR / "param_grid.png"}')
plt.show()

## 5.7 Observation Density Scaling

How does recovery accuracy scale with the number of observations? We re-run the inverse at $N_{\text{obs}} \in \{100, 200, 400, 800, 1600\}$, all at the canonical operating point $(D^*, k^*) = (0.32, 0.020)$ with $\eta = 2\%$ noise and seed $1234$. Each level is also fitted with the FD reference estimator, whose error follows the noise-limited (Cramér–Rao-like) $1/\sqrt{N_{\text{obs}}}$ trend; comparing the PINN against it shows whether the PINN tracks the noise floor. With a single noise realization per level, individual points fluctuate.

Runtime: ~20 min on A100.

In [ ]:
# ---- Observation density scaling ---------------------------------------
N_OBS_SWEEP = [100, 200, 400, 800, 1600]

print('=' * 72)
print(f'  Observation density sweep over {len(N_OBS_SWEEP)} levels: {N_OBS_SWEEP}')
print('=' * 72)

obs_density_results = []
od_t0 = time.time()

for n_obs in N_OBS_SWEEP:
    print(f'\n--- N_obs = {n_obs} ---')
    ox, oy, ot, oC = make_noisy_observations(n_obs=n_obs, noise_pct=2.0,
                                              rng=np.random.default_rng(SEED))
    obs_n = {key: jnp.asarray(np.asarray(v, dtype=np.float64))
             for key, v in dict(x_d=ox, y_d=oy, t_d=ot, C_d=oC).items()}
    D_n_rec, k_n_rec, _ = train_inverse_once(obs_n, seed=SEED, write_variables_dat=False)
    ref_n = fd_fit(ox, oy, ot, oC, noise_pct=2.0)
    D_n_err = 100.0 * abs(D_n_rec - D_TRUE) / D_TRUE
    k_n_err = 100.0 * abs(k_n_rec - K_TRUE) / K_TRUE
    obs_density_results.append({
        'n_obs': n_obs,
        'D_rec': D_n_rec, 'k_rec': k_n_rec,
        'D_rel_err_pct': D_n_err, 'k_rel_err_pct': k_n_err,
        'D_ref': ref_n['D'], 'k_ref': ref_n['k'],
        'D_ref_rel_err_pct': 100.0 * abs(ref_n['D'] - D_TRUE) / D_TRUE,
        'k_ref_rel_err_pct': 100.0 * abs(ref_n['k'] - K_TRUE) / K_TRUE,
    })
    elapsed = (time.time() - od_t0) / 60.0
    print(f'  -> N={n_obs}: PINN D = {D_n_rec:.4f} (|err|={D_n_err:.2f}%), '
          f'k = {k_n_rec:.5f} (|err|={k_n_err:.2f}%) | FD ref D = {ref_n["D"]:.4f}, '
          f'k = {ref_n["k"]:.5f}, elapsed = {elapsed:.1f} min')

print('\n=== Observation Density Scaling (PINN | FD reference) ===')
print(f'  {"N_obs":>6}  {"D":>8}  {"|D err|%":>9} {"ref%":>6}  {"k":>8}  {"|k err|%":>9} {"ref%":>6}')
for r in obs_density_results:
    print(f'  {r["n_obs"]:>6d}  {r["D_rec"]:>8.4f}  {r["D_rel_err_pct"]:>8.2f}% '
          f'{r["D_ref_rel_err_pct"]:>5.2f}%  {r["k_rec"]:>8.5f}  '
          f'{r["k_rel_err_pct"]:>8.2f}% {r["k_ref_rel_err_pct"]:>5.2f}%')

N_arr  = np.array([r['n_obs'] for r in obs_density_results])
D_errs = np.array([r['D_rel_err_pct'] for r in obs_density_results])
k_errs = np.array([r['k_rel_err_pct'] for r in obs_density_results])
D_ref_errs = np.array([r['D_ref_rel_err_pct'] for r in obs_density_results])
k_ref_errs = np.array([r['k_ref_rel_err_pct'] for r in obs_density_results])

fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(N_arr, D_errs, 'ro-', markersize=8, label='PINN: D error')
ax.loglog(N_arr, k_errs, 'b^-', markersize=8, label='PINN: k error')
ax.loglog(N_arr, D_ref_errs, 'ro--', mfc='none', alpha=0.7, label='FD reference: D error')
ax.loglog(N_arr, k_ref_errs, 'b^--', mfc='none', alpha=0.7, label='FD reference: k error')
ax.set_xlabel(r'Number of observations $N_{\mathrm{obs}}$')
ax.set_ylabel(r'Relative recovery error (%)')
ax.set_title(f'Observation density scaling at $\\eta = 2\\%$ noise')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'obs_density_scaling.png', dpi=200, bbox_inches='tight')
print(f'\nSaved {FIG_DIR / "obs_density_scaling.png"}')
plt.show()

## 5.8 Comparison with Levenberg–Marquardt Baseline

The natural classical baseline for this inverse problem is to fit the closed-form analytical solution (infinite-domain 2D Gaussian) to the noisy observations via Levenberg–Marquardt non-linear least-squares. This is the naive baseline: the analytical solution ignores the zero-flux walls, so it is misspecified for bounded-domain data. The FD reference estimator (Section 5.4) is the correctly specified counterpart and gives the noise-limited floor.

Runtime: <1 minute (scipy on CPU).

In [ ]:
# ---- Levenberg-Marquardt baseline using the analytical solution -------
from scipy.optimize import least_squares

def analytical_model(theta, x, y, t):
    """2D Gaussian + exponential decay (analytical, infinite-domain)."""
    D, k = theta
    s2 = SIGMA ** 2 + 2.0 * D * t
    return C0 * SIGMA ** 2 / s2 * np.exp(-(x ** 2 + y ** 2) / (2.0 * s2)) * np.exp(-k * t)

def residuals_fn(theta, x_obs, y_obs, t_obs, C_obs):
    return analytical_model(theta, x_obs, y_obs, t_obs) - C_obs

# Use the same canonical (2%, 400-obs) observations as the primary inverse run
x_obs_np = np.asarray(obs_x)
y_obs_np = np.asarray(obs_y)
t_obs_np = np.asarray(obs_t)
C_obs_np = np.asarray(obs_C)

# Same offset initial guess as the PINN
theta0 = np.array([D_INIT, K_INIT])

t0 = time.time()
result_lm = least_squares(
    residuals_fn,
    theta0,
    args=(x_obs_np, y_obs_np, t_obs_np, C_obs_np),
    method='lm',
)
lm_time = time.time() - t0

D_lm, k_lm = float(result_lm.x[0]), float(result_lm.x[1])
D_lm_err = 100.0 * abs(D_lm - D_TRUE) / D_TRUE
k_lm_err = 100.0 * abs(k_lm - K_TRUE) / K_TRUE

print('=' * 72)
print('  Levenberg-Marquardt baseline (analytical model)')
print('=' * 72)
print(f'  Runtime: {lm_time*1000:.1f} ms')
print(f'  D_LM = {D_lm:.4f}  (true {D_TRUE}, |err| = {D_lm_err:.2f}%)')
print(f'  k_LM = {k_lm:.4f}  (true {K_TRUE}, |err| = {k_lm_err:.2f}%)')
print(f'  LM optimality: cost = {result_lm.cost:.4e}, nfev = {result_lm.nfev}')

print('\n=== LM (naive) vs PINN vs FD reference (oracle) ===')
print(f'                    LM (analytical)   PINN (seed {INVERSE_SEEDS[0]})   FD reference')
print(f'  D recovered:    {D_lm:>14.4f}   {D_rec:>14.4f}   {D_star:>12.4f}')
print(f'  D rel error:    {D_lm_err:>13.2f}%   {rel_D:>13.2f}%   {D_ref_err:>11.2f}%')
print(f'  k recovered:    {k_lm:>14.5f}   {k_rec:>14.5f}   {k_star:>12.5f}')
print(f'  k rel error:    {k_lm_err:>13.2f}%   {rel_k:>13.2f}%   {k_ref_err:>11.2f}%')
print(f'  Runtime:        {lm_time*1000:>12.0f} ms     {ADAM_ITERS//N_CHUNK + 1:>10d} chunks (~10 min)')
print()
print('  Caveat: LM uses the INFINITE-DOMAIN analytical solution, which is')
print('  only valid while the Gaussian pulse remains well inside [-L/2, L/2].')
print('  At late times the boundary reflections invalidate this approximation;')
print('  the PINN correctly models the bounded-domain dynamics (FD-validated).')

## 5.9 Architecture Ablation

To defend the choice of network architecture, we re-run the **forward problem** with three different MLP configurations: $[3, 32, 32, 32, 1]$ (compact), $[3, 64, 64, 64, 64, 1]$ (canonical, used elsewhere in this paper), and $[3, 128, 128, 128, 1]$ (wide). All other hyperparameters are held constant.

Runtime: ~10 min on A100 (3 forward trainings).

In [ ]:
# ---- Architecture ablation: forward L2 vs network size -----------------
ARCH_CONFIGS = [
    ([3, 32, 32, 32, 1],            'compact'),
    ([3, 64, 64, 64, 64, 1],        'canonical'),
    ([3, 128, 128, 128, 1],         'wide'),
]

def count_params(layers):
    return sum(layers[i] * layers[i+1] + layers[i+1] for i in range(len(layers)-1))

print('=' * 72)
print(f'  Architecture ablation over {len(ARCH_CONFIGS)} forward configurations')
print('=' * 72)

arch_results = []
arch_t0 = time.time()

for arch_layers, arch_name in ARCH_CONFIGS:
    n_params = count_params(arch_layers)
    print(f'\n--- {arch_name}: {arch_layers}  ({n_params} params) ---')

    rngs_a = nnx.Rngs(SEED)
    model_a = MLP(arch_layers, rngs=rngs_a)
    opt_a = nnx.Optimizer(model_a, optax.adam(LR), wrt=nnx.Param)

    @nnx.jit(static_argnames=('n_chunk',))
    def adam_chunk_arch(model, optimizer, pts, n_chunk):
        last_loss = jnp.zeros(())
        for _ in range(n_chunk):
            loss, grads = nnx.value_and_grad(forward_loss)(model, pts)
            optimizer.update(model, grads)
            last_loss = loss
        return last_loss

    n_chunks = ADAM_ITERS // N_CHUNK
    t_arch = time.time()
    for c in range(n_chunks):
        adam_chunk_arch(model_a, opt_a, pts, N_CHUNK)
    adam_time = time.time() - t_arch

    # L-BFGS phase
    gdef_a, state_a = nnx.split(model_a)
    state_a = jax.tree.map(
        lambda x: x.astype(jnp.float64) if hasattr(x, 'dtype') and jnp.issubdtype(x.dtype, jnp.floating) else x,
        state_a,
    )
    def loss_a(params):
        return forward_loss(nnx.merge(gdef_a, params), pts)
    t_lb = time.time()
    res_a = LBFGS(fun=loss_a, maxiter=LBFGS_ITERS, tol=1e-9).run(state_a)
    lbfgs_time = time.time() - t_lb
    model_a = nnx.merge(gdef_a, res_a.params)

    # Evaluate forward L2 vs FD reference on the same grid
    @jax.jit
    def predict_arch(model_state, xyt_batch):
        m = nnx.merge(gdef_a, model_state)
        return jax.vmap(lambda row: m(row)[0])(xyt_batch)

    _, state_eval = nnx.split(model_a)
    XYT_full = np.stack([Xg.ravel(), Yg.ravel(), Tg.ravel()], axis=1)
    C_pred_arch = []
    for i in range(0, XYT_full.shape[0], 50_000):
        batch = jnp.asarray(XYT_full[i:i+50_000])
        C_pred_arch.append(np.asarray(predict_arch(state_eval, batch)))
    C_pred_arch = np.concatenate(C_pred_arch).reshape(nx_eval, ny_eval, nt_eval)
    err_fd_a = 100.0 * np.linalg.norm(C_pred_arch - C_fd) / np.linalg.norm(C_fd)

    arch_results.append({
        'name': arch_name,
        'layers': arch_layers,
        'n_params': n_params,
        'L2_fd_pct': float(err_fd_a),
        'adam_time_s': float(adam_time),
        'lbfgs_time_s': float(lbfgs_time),
        'total_time_s': float(adam_time + lbfgs_time),
    })
    print(f'  -> L2 vs FD = {err_fd_a:.3f}%, '
          f'Adam = {adam_time:.1f}s, LBFGS = {lbfgs_time:.1f}s, '
          f'total = {adam_time + lbfgs_time:.1f}s')

print(f'\nTotal architecture-ablation time: {(time.time() - arch_t0)/60:.1f} min')

print('\n=== Architecture Ablation Summary ===')
print(f'  {"Config":>10}  {"Params":>8}  {"L2 vs FD":>10}  {"Train (s)":>10}')
for r in arch_results:
    print(f'  {r["name"]:>10}  {r["n_params"]:>8d}  '
          f'{r["L2_fd_pct"]:>9.3f}%  {r["total_time_s"]:>10.1f}')

## 6. Save `metrics.json` and auto-fill the manuscript

If `AUTO_FILL = True` in Cell 4, the next two cells:
1. Save the computed metrics to `figures/metrics.json`
2. Prompt you to upload `B3_Dopamine_PINN_Paper.tex`
3. Replace the `[TBD]` markers with the actual computed values
4. Download the filled `.tex`

In [ ]:
metrics = {
    'dimension':       '2D',
    'implementation':  'JAX/Flax NNX + Optax + jaxopt + Blackjax',
    'true_params':     {'D': D_TRUE, 'k': K_TRUE},
    'hyperparameters': {
        'layers':      LAYERS,
        'activation':  'tanh',
        'adam_iters':  ADAM_ITERS,
        'lbfgs_iters': LBFGS_ITERS,
        'n_domain':    N_DOMAIN,
        'noise_pct':   2.0,
        'quick_mode':  QUICK,
        'sampling':    'LatinHypercube (interior); uniform (IC, BC)',
        'obs_time_window_ms': [0.5, 0.9 * T],
        'T_ms':        T,
        'input_scaling': 'none (raw x, y, t)',
    },
    'forward': {
        'L2_vs_analytical_pct': float(err_anal),
        'L2_vs_fd_pct':         float(err_fd),
        'per_snapshot':         per_snapshot,
    },
    'inverse': {
        'D_recovered':     D_rec, 'D_rel_error_pct': float(rel_D),
        'k_recovered':     k_rec, 'k_rel_error_pct': float(rel_k),
        'n_observations':  int(N_OBS), 'noise_pct': 2.0,
        'seed':            int(INVERSE_SEEDS[0]),
    },
    'inverse_multiseed': {
        'seeds':                  [int(s) for s in INVERSE_SEEDS],
        'D_per_seed':             per_seed_D.tolist(),
        'k_per_seed':             per_seed_k.tolist(),
        'D_mean':                 D_mean,
        'D_std':                  D_std,
        'k_mean':                 k_mean,
        'k_std':                  k_std,
        'D_rel_error_mean_pct':   D_err_mean,
        'D_rel_error_std_pct':    D_err_std,
        'k_rel_error_mean_pct':   k_err_mean,
        'k_rel_error_std_pct':    k_err_std,
    },
    'reference_estimator': {
        'method':          'Nonlinear least squares with the bounded-domain FD solver '
                           '(oracle; same model as the data generator)',
        'D_recovered':     D_star,
        'k_recovered':     k_star,
        'D_rel_error_pct': D_ref_err,
        'k_rel_error_pct': k_ref_err,
        'n_fd_solves':     fd_ref['nfev'],
        'runtime_s':       float(fd_ref_time),
    },
    'uncertainty_quantification': {
        'method':              'Laplace approximation (Gauss-Newton Hessian) around the '
                               'FD-likelihood MAP; flat prior on (log D, log k)',
        'D_map':               D_star,
        'k_map':               k_star,
        'D_posterior_std':     D_post_std,
        'k_posterior_std':     k_post_std,
        'D_ci95':              [float(D_lo), float(D_hi)],
        'k_ci95':              [float(k_lo), float(k_hi)],
        'D_covers_truth':      bool(D_covers),
        'k_covers_truth':      bool(k_covers),
        'correlation_rho_Dk':  rho_Dk,
        'covariance_Dk':       np.asarray(Sigma_Dk).tolist(),
        'n_posterior_samples': int(n_post_samples),
    },
}

# Noise sweep
if NOISE_SWEEP and sweep_D.size > 0:
    metrics['noise_sweep'] = {
        'noise_levels_pct':         list(NOISE_LEVELS),
        'D_per_noise':              sweep_D.tolist(),
        'k_per_noise':              sweep_k.tolist(),
        'D_rel_error_pct_per_noise': sweep_D_err.tolist(),
        'k_rel_error_pct_per_noise': sweep_k_err.tolist(),
        'D_ref_per_noise':           sweep_D_ref.tolist(),
        'k_ref_per_noise':           sweep_k_ref.tolist(),
        'D_ref_rel_error_pct_per_noise': sweep_D_ref_err.tolist(),
        'k_ref_rel_error_pct_per_noise': sweep_k_ref_err.tolist(),
        'seed_used':                int(SEED),
        'n_observations':           int(N_OBS),
    }

# Parameter recovery grid
try:
    metrics['parameter_grid'] = {
        'combinations':   param_grid_results,
        'noise_pct':      2.0,
        'n_observations': int(N_OBS),
        'seed':           int(SEED),
    }
except NameError:
    print('(parameter_grid block skipped)')

# Observation density scaling
try:
    metrics['obs_density_sweep'] = {
        'n_obs_levels': N_OBS_SWEEP,
        'results':      obs_density_results,
        'noise_pct':    2.0,
        'seed':         int(SEED),
    }
except NameError:
    print('(obs_density_sweep block skipped)')

# LM baseline
try:
    metrics['lm_baseline'] = {
        'method':          'Levenberg-Marquardt fit of analytical 2D Gaussian',
        'D_recovered':     D_lm,
        'k_recovered':     k_lm,
        'D_rel_error_pct': D_lm_err,
        'k_rel_error_pct': k_lm_err,
        'runtime_ms':      float(lm_time * 1000),
        'cost':            float(result_lm.cost),
        # nfev, not njev: SciPy returns njev=None for method='lm'
        'n_function_evals': int(result_lm.nfev),
    }
except NameError:
    print('(lm_baseline block skipped)')

# Architecture ablation
try:
    metrics['architecture_ablation'] = {
        'configurations': arch_results,
        'note': 'Forward-problem L2 error vs FD reference for three MLP sizes',
    }
except NameError:
    print('(architecture_ablation block skipped)')

with open(FIG_DIR / 'metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=2, default=float)
print(f'Saved {FIG_DIR / "metrics.json"}')
print()
print(json.dumps(metrics, indent=2, default=float))

In [ ]:
if AUTO_FILL:
    try:
        from google.colab import files
        print('Please select B3_Dopamine_PINN_Paper.tex to upload...')
        uploaded = files.upload()
        tex_name = next(iter(uploaded.keys()))
    except ImportError:
        tex_name = 'B3_Dopamine_PINN_Paper.tex'
        if not Path(tex_name).exists():
            raise FileNotFoundError(f'{tex_name} not found in working directory.')

    text = Path(tex_name).read_text()

    # Use t = 1 ms per-snapshot L2 vs analytical as the headline
    # (consistent with the manuscript phrasing).
    L2_anal_headline = per_snapshot['t_1']['L2_anal_pct']
    max_rec_err = max(rel_D, rel_k)

    # Replacement order matters: do the precise patterns first.
    replacements = [
        # Forward L2 vs FD (full window) -- the headline accuracy claim
        (r'L2 relative error of \\textbf\{\[TBD\]\}\\% compared with a finite-difference',
         f'L2 relative error of {err_fd:.2f}\\% compared with a finite-difference'),
        # Forward L2 vs analytical (t = 1 ms)
        (r'L2 relative error of \\textbf\{\[TBD\]\}\\% compared with\s+the analytical',
         f'L2 relative error of {L2_anal_headline:.2f}\\% compared with the analytical'),
        # Conclusion: L2 relative error of [TBD]% against the analytical
        (r'L2 relative error of \\textbf\{\[TBD\]\}\\% against the analytical',
         f'L2 relative error of {L2_anal_headline:.2f}\\% against the analytical'),
        # Recovered D, k values
        (r'\$D =\$ \\textbf\{\[TBD\]\}', f'$D =$ {D_rec:.4f}'),
        (r'\$k =\$ \\textbf\{\[TBD\]\}', f'$k =$ {k_rec:.4f}'),
        # Abstract: relative errors of [TBD]% and [TBD]%
        (r'relative errors of \\textbf\{\[TBD\]\}\\% and \\textbf\{\[TBD\]\}\\%',
         f'relative errors of {rel_D:.2f}\\% and {rel_k:.2f}\\%'),
        # Conclusion: relative errors below [TBD]%
        (r'with relative\s+errors below \\textbf\{\[TBD\]\}\\%',
         f'with relative errors below {max_rec_err:.2f}\\%'),
        # Conclusion: even at [TBD]% observational noise
        (r'even at \\textbf\{\[TBD\]\}\\% observational noise',
         f'even at 2.00\\% observational noise'),
        # Table 3 caption: within [TBD] percent
        (r'within \\textbf\{\[TBD\]\} percent',
         f'within {max_rec_err:.1f}\\% '),
    ]
    n_subs = 0
    for pat, repl in replacements:
        text, n = re.subn(pat, repl, text)
        n_subs += n

    # Per-snapshot table 2 cells: replace 8 cells in order with t=1, 5, 10, 20 vs anal/FD
    snap_pairs = [(per_snapshot[f't_{tt}']['L2_anal_pct'], per_snapshot[f't_{tt}']['L2_fd_pct'])
                  for tt in (1, 5, 10, 20)]
    snap_iter = iter([f'{v:.3f}' for pair in snap_pairs for v in pair])
    def _next_snap(m):
        try: return next(snap_iter) + '\\%'
        except StopIteration: return m.group(0)
    # Only substitute table cells in the rows starting with $t = N$
    def _row_sub(m):
        prefix = m.group(1)
        cells = m.group(2)
        new = re.sub(r'\\textbf\{\[TBD\]\}\\%', _next_snap, cells)
        return prefix + new
    text = re.sub(r'(\$t = \d+\$\s+&\s+)((?:\\textbf\{\[TBD\]\}\\%\s*&?\s*)+\\\\)',
                  _row_sub, text)

    # Table 3 row: 2% & [TBD] & [TBD]% & [TBD] & [TBD]%
    text = re.sub(
        r'\$2\\%\$\s+&\s+\\textbf\{\[TBD\]\}\s+&\s+\\textbf\{\[TBD\]\}\\%'
        r'\s+&\s+\\textbf\{\[TBD\]\}\s+&\s+\\textbf\{\[TBD\]\}\\%',
        f'$2\\%$    & {D_rec:.4f} & {rel_D:.2f}\\% & {k_rec:.4f} & {rel_k:.2f}\\%',
        text,
    )

    out_name = tex_name.replace('.tex', '_filled.tex')
    Path(out_name).write_text(text)
    remaining = len(re.findall(r'\\textbf\{\[TBD\]\}', text))
    print(f'Substitutions applied. [TBD] markers remaining: {remaining}')
    print(f'Saved {out_name}')
    try:
        from google.colab import files as colab_files
        colab_files.download(out_name)
    except ImportError:
        print('(Not on Colab - filled .tex remains in working directory.)')
else:
    print('Skipping auto-fill (set AUTO_FILL=True to enable).')

## 7. Download all artifacts

In [ ]:
import zipfile

archive = 'dopamine_PINN_2D_artifacts.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(FIG_DIR.iterdir()):
        if p.is_file():
            zf.write(p, arcname=p.name)
            print(f'  + {p.name}  ({p.stat().st_size // 1024} KB)')
print(f'\nWrote {archive} ({Path(archive).stat().st_size // 1024} KB)')

try:
    from google.colab import files as colab_files
    colab_files.download(archive)
except ImportError:
    print('(Not on Colab - archive remains in working directory.)')